In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key="api_key",
)

print("LLM configured", llm)


LLM configured metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.3.18', 'langchain-openai': '1.6.0'}} profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True} client=<openai.resources.chat.completions.completions.Completions object at 0x0000017B6267A510> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000017B6267AF90> root_client=<openai.OpenAI object at 0x0000017B61DE46E0

Zero-shot prompting

In [11]:
response = llm.invoke("Classify the sentiment of this review as positive, negative, or neutral:"
                      "'The delivery was late but the product quality is excellent.'")
print(response.content)

The sentiment of the review is mixed, but it leans towards positive due to the emphasis on the excellent product quality. Therefore, it can be classified as **neutral** because it contains both a negative aspect (late delivery) and a positive aspect (excellent product quality).


Few-shot prompting

In [5]:
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate

examples = [
    {"review": "Amazing product, fast shipping!", "sentiment": "positive"},
    {"review": "It broke after two days, terrible.", "sentiment": "negative"},
    {"review": "It's okay, does what it says.", "sentiment": "neutral"},
]

example_prompt = PromptTemplate(
    input_variables=["review", "sentiment"],
    template="Review: {review}\nSentiment: {sentiment}"
)

few_shot_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    prefix="Classify the sentiment of each review as positive, negative, or neutral.",
    suffix="Review: {input}\nSentiment:",
    input_variables=["input"],
)

formatted_prompt = few_shot_prompt.format(input="The delivery was late but the product quality is excellent.")

print(formatted_prompt)

response = llm.invoke(formatted_prompt)
print("response: ", response.content)



Classify the sentiment of each review as positive, negative, or neutral.

Review: Amazing product, fast shipping!
Sentiment: positive

Review: It broke after two days, terrible.
Sentiment: negative

Review: It's okay, does what it says.
Sentiment: neutral

Review: The delivery was late but the product quality is excellent.
Sentiment:
response:  positive


In [7]:
prompt = """Classify the sentiment of each review as positive, negative, or neutral.

Review: Amazing product, fast shipping!
Sentiment: positive

Review: It broke after two days, terrible.
Sentiment: negative

Review: It's okay, does what it says.
Sentiment: neutral

Review: {input}
Sentiment:
"""

formatted_prompt = prompt.format(input="The delivery was late but the product quality is excellent.")

response = llm.invoke(formatted_prompt)

print("response: ", response.content)

response:  Sentiment: positive


Chain-of-thought prompting

In [ ]:
response = llm.invoke("A store has 3 boxes of 24 apples each. They sell 50 apples. how many are left?")
print("Direct answer:", response.content)

Direct answer: First, we calculate the total number of apples in the store. Since there are 3 boxes with 24 apples each, we multiply:

\[
3 \times 24 = 72 \text{ apples}
\]

Next, we need to subtract the number of apples sold from the total. The store sells 50 apples:

\[
72 - 50 = 22 \text{ apples}
\]

Therefore, the number of apples left in the store is 22.


In [9]:
response = llm.invoke(
    "A store has 3 boxes of 24 apples each. They sell 50 apples. how many are left?"
    "Think step by step before giving the final answer."
)
print("Chain-of-thought answer:", response.content)

Chain-of-thought answer: Let's break down the problem step by step.

1. **Calculate the total number of apples**: 
   The store has 3 boxes, and each box contains 24 apples. 
   \[
   \text{Total apples} = 3 \text{ boxes} \times 24 \text{ apples/box} = 72 \text{ apples}
   \]

2. **Determine how many apples are sold**:
   The store sells 50 apples.

3. **Calculate the remaining apples**:
   To find out how many apples are left, subtract the number of apples sold from the total number of apples.
   \[
   \text{Remaining apples} = \text{Total apples} - \text{Apples sold} = 72 \text{ apples} - 50 \text{ apples} = 22 \text{ apples}
   \]

So, after selling 50 apples, the store has **22 apples left**.


Role / Persona Prompting

In [10]:
from langchain_core.messages import HumanMessage, SystemMessage

messages = [
    SystemMessage(content="You are a strict code reviewer. Be blunt and point out every issue, no matter how small."),
    HumanMessage(content="def add(a,b): return a+b")
]

response = llm.invoke(messages)
print("strict reviewer: ", response.content)

strict reviewer:  Let's break down the issues with your code:

1. **Function Definition Style**: While your function is technically valid, using a single line returns is usually discouraged for more complex functions. This style hinders readability and maintainability. Consider defining the function with a block:

   ```python
   def add(a, b):
       return a + b
   ```

2. **Lack of Type Annotations**: You should include type annotations for better clarity and to help document your function’s expected inputs and outputs.

   ```python
   def add(a: Union[int, float], b: Union[int, float]) -> Union[int, float]:
   ```

   You'd need to import `Union` from `typing` for this.

3. **Documentation**: There is no docstring for the function. You should include a docstring to describe what the function does, its parameters, and its return value.

   ```python
   def add(a: Union[int, float], b: Union[int, float]) -> Union[int, float]:
       """
       Adds two numbers together.

       Para

In [ ]:
messages = [
    SystemMessage(content="You are a friendly, encouraging coding mentor for beginners."),
    HumanMessage(content="def add(a,b): return a+b")
]

response = llm.invoke(messages)
print("friendly mentor: ", response.content)

friendly mentor:  Great job on writing a simple addition function! The function `add` you’ve created takes two parameters, `a` and `b`, and returns their sum. This is a solid foundation for understanding how functions work in Python.

Here's a bit of extra information you might find useful:

1. **Function Definition**: You've successfully defined a function using the `def` keyword. 
2. **Parameters**: The parameters `a` and `b` allow you to pass in values when you call the function.
3. **Return Statement**: The `return` keyword sends back the result of the addition to the caller.

Would you like to see how to use this function, or perhaps explore more advanced concepts like type hints or error handling? Keep up the great work!


In [13]:
from langchain_core.prompts import ChatPromptTemplate

template = ChatPromptTemplate.from_messages([
    ("system", "You are a {role}. Answer in a {tone} tone."),
    ("human", "{question}")
])

support_prompt = template.partial(role="customer support agent", tone="polite and professional")

chain = support_prompt | llm
response = chain.invoke({"question": "Why hasn't my order arrived yet?"})
print("response: ", response.content)

response:  I apologize for any inconvenience you’re experiencing with your order delivery. To assist you better, could you please provide me with your order number? This will help me investigate the status of your order and give you more information. Thank you for your patience!


In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory

history = InMemoryChatMessageHistory()
history.add_user_message("My name is Chetan")
history.add_ai_message("Nice to meet you, Chetan!")
history.add_user_message("What is my name?")

response = llm.invoke(history.messages)
print("response: ", response.content)

response:  Your name is Chetan.


In [24]:
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

base_chain = prompt | llm

store = {}

def get_history(session_id: str):

    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    
    return store[session_id]

chain_with_memory = RunnableWithMessageHistory(
    base_chain,
    get_history,
    input_messages_key="input",
    history_messages_key="history",
)

config = {"configurable": {"session_id": "demo-session"}}

response = chain_with_memory.invoke({"input": "My name is chetan"}, config=config)
print("response1: ", response.content)

response = chain_with_memory.invoke({"input": "What is my name?"}, config=config)
print("response2: ", response.content)

e:\class\ai-class\backend\venv\Lib\site-packages\IPython\core\interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


response1:  Hello, Chetan! How can I assist you today?
response2:  Your name is Chetan. How can I help you today?


In [25]:
config_alice = {"configurable": {"session_id": "alice-session"}}
config_john = {"configurable": {"session_id": "john-session"}}

chain_with_memory.invoke({"input": "My name is Alice"}, config=config_alice)
chain_with_memory.invoke({"input": "My name is John"}, config=config_john)

alice_response = chain_with_memory.invoke({"input": "what is my name?"}, config=config_alice)
john_response = chain_with_memory.invoke({"input": "what is my name?"}, config=config_john)

print("response3: ", alice_response.content)
print("response4: ", john_response.content)

response3:  Your name is Alice! How can I help you today?
response4:  Your name is John! How can I help you today, John?


In [27]:
print("Active session:", list(store.keys()))

print("Alice's full session history")
for msg in store["alice-session"].messages:
    print(f"{msg.type}: {msg.content}")

Active session: ['demo-session', 'alice-session', 'john-session']
Alice's full session history
human: My name is Alice
ai: Nice to meet you, Alice! How can I assist you today?
human: what is my name?
ai: Your name is Alice! How can I help you today?


In [ ]:
from langchain_core.messages import trim_messages, HumanMessage, AIMessage, SystemMessage

long_history = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content="What's the capital of France?"),
    AIMessage(content="The capital of France is Paris."),
    HumanMessage(content="What's the capital of Japan?"),
    AIMessage(content="The capital of Japan is Tokyo."),
    HumanMessage(content="What's the capital of Italy?"),
    AIMessage(content="The capital of Italy is Rome."),
    HumanMessage(content="What was the first question I asked?"),
]

# 128k input
# 16k ouput

trimmed = trim_messages(
    long_history,
    max_tokens=50000,
    strategy="last",
    token_counter=llm,
    include_system=True,
    start_on="human",
)

for msg in trimmed:
    print(f"{msg.type}: {msg.content}")

system: You are a helpful assistant.
human: What's the capital of Japan?
ai: The capital of Japan is Tokyo.
human: What's the capital of Italy?
ai: The capital of Italy is Rome.
human: What was the first question I asked?
